In [1]:
import pandas as pd
import sqlite3

# Load the cleaned dataset
df = pd.read_csv("../data/processed/dublinbikes_with_weather.csv")

# Create a SQLite database file and load the dataframe into it as a table
conn = sqlite3.connect("../data/processed/dublinbikes.db")
df.to_sql("bikes", conn, if_exists="replace", index=False)

print("Table loaded. Row count check:")
print(pd.read_sql("SELECT COUNT(*) as row_count FROM bikes", conn))

Table loaded. Row count check:
   row_count
0     322307


In [2]:
#Top 10 empty-morning stations

query1 = """
SELECT 
    NAME,
    ROUND(100.0 * SUM(CASE WHEN AVAILABLE_BIKES = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_empty_morning
FROM bikes
WHERE hour IN (8, 9)
GROUP BY NAME
ORDER BY pct_empty_morning DESC
LIMIT 10
"""

result1 = pd.read_sql(query1, conn)
print(result1)

                           NAME  pct_empty_morning
0                    BROADSTONE              55.08
1             CUSTOM HOUSE QUAY              49.15
2        HEUSTON BRIDGE (SOUTH)              39.41
3              HARDWICKE STREET              37.71
4               PORTOBELLO ROAD              34.32
5  GRANGEGORMAN LOWER (CENTRAL)              33.90
6                FRANCIS STREET              31.45
7            BLESSINGTON STREET              28.39
8          MOUNTJOY SQUARE EAST              27.54
9            PORTOBELLO HARBOUR              24.58


In [3]:
#Weekday vs weekend average

query2 = """
SELECT 
    is_weekend,
    ROUND(AVG(AVAILABLE_BIKES), 2) AS avg_available_bikes
FROM bikes
GROUP BY is_weekend
"""

result2 = pd.read_sql(query2, conn)
print(result2)

   is_weekend  avg_available_bikes
0           0                11.71
1           1                12.04


In [4]:
query1_clean = """
SELECT 
    NAME,
    ROUND(100.0 * SUM(CASE WHEN AVAILABLE_BIKES = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_empty_morning
FROM bikes
WHERE hour IN (8, 9) AND NAME NOT LIKE '%TEST%'
GROUP BY NAME
ORDER BY pct_empty_morning DESC
LIMIT 10
"""

result1_clean = pd.read_sql(query1_clean, conn)
print(result1_clean)

                           NAME  pct_empty_morning
0                    BROADSTONE              55.08
1             CUSTOM HOUSE QUAY              49.15
2        HEUSTON BRIDGE (SOUTH)              39.41
3              HARDWICKE STREET              37.71
4               PORTOBELLO ROAD              34.32
5  GRANGEGORMAN LOWER (CENTRAL)              33.90
6                FRANCIS STREET              31.45
7            BLESSINGTON STREET              28.39
8          MOUNTJOY SQUARE EAST              27.54
9            PORTOBELLO HARBOUR              24.58


In [5]:
with open("../scripts/analysis_queries.sql", "w") as f:
    f.write("-- Top 10 stations most often empty during morning peak (8-9am)\n")
    f.write(query1_clean.strip() + ";\n\n")
    f.write("-- Average bikes available: weekday vs weekend\n")
    f.write(query2.strip() + ";\n")

print("Saved SQL queries to scripts/analysis_queries.sql")

Saved SQL queries to scripts/analysis_queries.sql
